In [1]:
import torch
print("hello, torch!")
if not torch.cuda.is_available():
    raise EnvironmentError(
        "❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name}!")
print(f"✅ VRAM: {vram_gb:.1f} GB")

hello, torch!
✅ GPU: Tesla T4!
✅ VRAM: 15.6 GB


In [ ]:
!pip install -q unsloth datasets rouge_score

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from huggingface_hub import login
import json, re, random

login(token="YOUR_HF_TOKEN_HERE")


import os
from google.colab import drive
drive.mount('/content/drive')

MODEL_ID       = "unsloth/gemma-4-e2b-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 256
LORA_RANK      = 16
SYSTEM_PROMPT  = (
    "You are SurvivalGuide, an expert survival assistant. "
    "You provide clear, practical, actionable advice on survival situations "
    "including power outages, water shortages, natural disasters, wilderness "
    "survival, first aid, and emergency preparedness. "
    "Your answers are concise, direct, and prioritise safety above all else."
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,   # QLoRA
)
print("✅ Model loaded")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    bias="none",
)
print("✅ LoRA adapters injected")

def parse_qa(filepath):
    with open(filepath, "r") as f:
        text = f.read()
    pattern = re.compile(r"Q:\s*(.+?)\nA:\s*(.+?)(?=\nQ:|\Z)", re.DOTALL)
    pairs = []
    for m in pattern.finditer(text):
        q, a = m.group(1).strip(), m.group(2).strip()
        if len(q) > 10 and len(a) > 20:
            pairs.append({"question": q, "answer": a})
    return pairs

def format_example(pair):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": pair["question"]},
        {"role": "assistant", "content": pair["answer"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

pairs = parse_qa("/content/drive/MyDrive/LM/instructions_upsampled.txt")
random.seed(42)
random.shuffle(pairs)
split = int(len(pairs) * 0.9)

train_data = Dataset.from_dict({"text": [format_example(p) for p in pairs[:split]]})
eval_data  = Dataset.from_dict({"text": [format_example(p) for p in pairs[split:]]})
print(f"✅ Train: {len(train_data)} | Eval: {len(eval_data)}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=SFTConfig(
    output_dir="adapters/survival",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,           
    bf16=False,          
    gradient_checkpointing=True,
    max_seq_length=MAX_SEQ_LENGTH,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    dataset_text_field="text",
),
)
trainer.train()
print("✅ Training complete")

model.save_pretrained_gguf(
    "survival_gguf",
    tokenizer,
    quantization_method="q4_k_m"
)
print("✅ GGUF saved — download survival_gguf/ for phone deployment")

import shutil
shutil.make_archive("survival_gguf", "zip", "survival_gguf")
files.download("survival_gguf.zip")

Mounted at /content/drive
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 5/5 [05:19<00:00, 63.99s/it] 


Unsloth: Regenerating safetensors index...
Unsloth: Merge process complete. Saved to `/content/survival_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...


RuntimeError: Unsloth: GGUF conversion failed: Unsloth: Failed to convert text model to GGUF with command `/usr/bin/python3 /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile gemma-4-e2b-it.F16.gguf --outtype f16 --split-max-size 50G survival_gguf`: Command '['/usr/bin/python3', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py', '--outfile', 'gemma-4-e2b-it.F16.gguf', '--outtype', 'f16', '--split-max-size', '50G', 'survival_gguf']' died with <Signals.SIGKILL: 9>.

In [4]:
import shutil
shutil.copytree("/content/survival_gguf", "/content/drive/MyDrive/LM/survival_backup", dirs_exist_ok=True)

'/content/drive/MyDrive/LM/survival_backup'

In [4]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=SFTConfig(
    output_dir="adapters/survival",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,              # ← T4 uses fp16, not bf16
    bf16=False,             # ← explicitly off
    gradient_checkpointing=True,
    max_seq_length=MAX_SEQ_LENGTH,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    dataset_text_field="text",
),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2367 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/264 [00:00<?, ? examples/s]

In [5]:
trainer.train()
print("✅ Training complete")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,367 | Num Epochs = 3 | Total steps = 888
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss,Validation Loss
100,1.689483,4.908744
200,1.247136,4.170811
300,0.954306,3.758711
400,0.707134,3.747104
500,0.516203,3.925354
600,0.360475,4.132038
700,0.201361,4.286222
800,0.166990,4.407404
888,0.151718,4.423903


Unsloth: Restored added_tokens_decoder metadata in adapters/survival/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in adapters/survival/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in adapters/survival/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in adapters/survival/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in adapters/survival/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in adapters/survival/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in adapters/survival/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in adapters/survival/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in adapters/survival/checkpoint-888/tokenizer_config.json.


✅ Training complete


In [1]:
model.save_pretrained_gguf(
    "survival_gguf",
    tokenizer,
    quantization_method="q4_k_m"
)
print("✅ GGUF saved — download survival_gguf/ for phone deployment")

NameError: name 'model' is not defined

In [ ]:
type(model)